# 🎬 YouTube Downloader Server

Загружает видео (720p) на Google Drive с оригинальным названием, а также субтитры (SRT+TXT) и превью (320x180).
Автоматический публичный URL через Cloudflare Tunnel.

## ✅ Features:
- **Видео 720p** → сохраняется на Google Drive с названием видео + ID
- **Субтитры** (SRT + TXT) – английские (обычные или автоматические) с поддержкой всех вариантов (en, en-US, en-GB, en-UK, en-AU, en-CA)
- **Превью** 320×180
- **Обработка канала** – скачивает последние N видео с канала или плейлиста, каждое в отдельную папку
- **title.txt** – внутри каждой папки сохраняется файл с названием видео
- CORS enabled
- Автоматический публичный URL через Cloudflare Tunnel

## 🚀 One‑click deployment

In [ ]:
# ============================================================================
# 🎬 YouTube Downloader + Google Drive (полная версия)
# ============================================================================
print("=" * 80)
print("📦 Установка зависимостей")
print("=" * 80)
!pip install -U yt-dlp flask requests -q

print("\n" + "=" * 80)
print("🔐 Монтирование Google Drive")
print("=" * 80)
from google.colab import drive
drive.mount('/content/drive')
import os
downloads_dir = '/content/drive/MyDrive/yt_downloads'
os.makedirs(downloads_dir, exist_ok=True)
print(f"✅ Видео будут сохраняться в: {downloads_dir}")

print("\n" + "=" * 80)
print("🚀 Запуск Flask‑сервера")
print("=" * 80)
import subprocess, threading, time, re, tempfile, shutil, uuid
from flask import Flask, request, jsonify, Response
import requests

app = Flask(__name__)

@app.after_request
def cors(response):
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type, X-Requested-With"
    return response

# ---------- Субтитры (универсальные: пробует все варианты английского) ----------
@app.route("/subtitles", methods=["GET", "OPTIONS"])
def get_subtitles():
    if request.method == 'OPTIONS':
        return '', 200
    url = request.args.get("url")
    lang = request.args.get("lang", "en")
    fmt = request.args.get("format", "srt").lower()
    autosubs = request.args.get("autosubs", "true").lower() == "true"
    if not url:
        return jsonify({"error": "No URL"}), 400

    # Все возможные варианты языка
    langs_to_try = [lang]
    if lang == 'en':
        langs_to_try.extend(['en-US', 'en-GB', 'en-UK', 'en-AU', 'en-CA'])

    time.sleep(2)

    with tempfile.TemporaryDirectory() as tmpdir:
        try:
            for try_lang in langs_to_try:
                output_template = os.path.join(tmpdir, f'subtitles.%(ext)s')
                cmd = [
                    'yt-dlp', '--skip-download',
                    '--write-subs' if not autosubs else '--write-auto-subs',
                    '--sub-lang', try_lang,
                    '--sub-format', fmt,
                    '--output', output_template,
                    url
                ]
                subprocess.run(cmd, capture_output=True, text=True)
                
                for ext in [f"{try_lang}.{fmt}", fmt, f"{try_lang}.vtt", "vtt"]:
                    fname = os.path.join(tmpdir, f'subtitles.{ext}')
                    if os.path.exists(fname):
                        with open(fname, 'r', encoding='utf-8') as f:
                            content = f.read()
                        if fmt == 'txt' and fname.endswith('.srt'):
                            lines = content.split('\n')
                            content = '\n'.join(l for l in lines if l.strip() and not l.strip().isdigit() and '-->' not in l)
                        resp = Response(content, mimetype='text/plain')
                        resp.headers["Content-Disposition"] = f"attachment; filename=subtitles.{fmt}"
                        return resp

            return jsonify({
                "error": "Subtitles not found",
                "message": f"No subtitles in language '{lang}' for this video"
            }), 404

        except Exception as e:
            return jsonify({"error": str(e)}), 500

# ---------- Картинка ----------
@app.route("/thumbnail", methods=["GET", "OPTIONS"])
def get_thumbnail():
    if request.method == 'OPTIONS':
        return '', 200
    url = request.args.get("url")
    if not url:
        return jsonify({"error": "No URL"}), 400
    m = re.search(r'(?:youtube\.com/watch\?v=|youtu\.be/)([^&\?/]+)', url)
    if not m:
        return jsonify({"error": "Invalid URL"}), 400
    vid = m.group(1)
    thumb_url = f"https://img.youtube.com/vi/{vid}/mqdefault.jpg"
    try:
        r = requests.get(thumb_url, timeout=10)
        if r.status_code != 200:
            thumb_url = f"https://img.youtube.com/vi/{vid}/hqdefault.jpg"
            r = requests.get(thumb_url, timeout=10)
            if r.status_code != 200:
                return jsonify({"error": "Thumbnail not found"}), 404
        resp = Response(r.content, mimetype='image/jpeg')
        resp.headers["Content-Disposition"] = "attachment; filename=thumbnail.jpg"
        return resp
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# ---------- Видео -> Google Drive (с именем видео) ----------
@app.route("/drive", methods=["GET", "OPTIONS"])
def upload_to_drive():
    if request.method == 'OPTIONS':
        return '', 200
    url = request.args.get("url")
    if not url:
        return jsonify({"error": "No YouTube URL"}), 400

    tmp_file = f"/tmp/video_{int(time.time())}.mp4"

    try:
        # Сначала получаем информацию о видео (название и ID)
        info_cmd = [
            'yt-dlp', '--skip-download',
            '--print', '%(title)s',
            '--print', '%(id)s',
            '--user-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            '--extractor-args', 'youtube:player_client=android,web',
            url
        ]
        info_proc = subprocess.run(info_cmd, capture_output=True, text=True, timeout=30)

        if info_proc.returncode != 0:
            video_title = f"video_{int(time.time())}"
            video_id = ""
        else:
            lines = info_proc.stdout.strip().split('\n')
            video_title = lines[0] if len(lines) > 0 else f"video_{int(time.time())}"
            video_id = lines[1] if len(lines) > 1 else ""

        safe_title = re.sub(r'[\\/*?:"<>|]', "", video_title)
        safe_title = safe_title.strip()[:80]

        if video_id:
            filename = f"{safe_title}_{video_id}.mp4"
        else:
            filename = f"{safe_title}.mp4"

        if len(filename) > 200:
            filename = filename[:180] + ".mp4"

        time.sleep(1)

        # Скачиваем видео
        cmd = [
            'yt-dlp',
            '-f', 'bestvideo[height<=720]+bestaudio/best[height<=720]',
            '--merge-output-format', 'mp4',
            '--user-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            '--extractor-args', 'youtube:player_client=android,web',
            '-o', tmp_file,
            url
        ]
        proc = subprocess.run(cmd, capture_output=True, timeout=240)
        if proc.returncode != 0:
            return jsonify({
                "error": "Download failed",
                "details": proc.stderr.decode()
            }), 500

        if not os.path.exists(tmp_file):
            return jsonify({"error": "File not found after download"}), 500

        # Сохраняем на Drive
        drive_path = os.path.join(downloads_dir, filename)

        if os.path.exists(drive_path):
            timestamp = int(time.time())
            name, ext = os.path.splitext(filename)
            filename = f"{name}_{timestamp}{ext}"
            drive_path = os.path.join(downloads_dir, filename)

        shutil.copy2(tmp_file, drive_path)
        os.remove(tmp_file)

        file_size = os.path.getsize(drive_path)
        file_size_mb = round(file_size / (1024 * 1024), 1)

        return jsonify({
            "success": True,
            "message": f"✅ Видео сохранено на Google Drive\n📁 Папка: yt_downloads\n📄 Имя: {filename}\n📊 Размер: {file_size_mb} MB",
            "filename": filename,
            "title": video_title,
            "size_mb": file_size_mb,
            "drive_link": "https://drive.google.com/drive/my-drive"
        })
    except subprocess.TimeoutExpired:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)
        return jsonify({"error": "Download timed out (video too large?)"}), 500
    except Exception as e:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)
        return jsonify({"error": str(e)}), 500

# ---------- Обработка канала (последние N видео) ----------
@app.route("/channel", methods=["GET", "OPTIONS"])
def process_channel():
    if request.method == 'OPTIONS':
        return '', 200
    channel_url = request.args.get("url")
    limit = int(request.args.get("limit", 10))
    if not channel_url:
        return jsonify({"error": "No channel URL provided"}), 400

    print(f"\n📺 Получение списка видео с канала: {channel_url}")
    cmd = [
        'yt-dlp', '--flat-playlist', '--print', '%(url)s',
        '--playlist-end', str(limit),
        channel_url
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    if proc.returncode != 0:
        return jsonify({"error": "Failed to get video list", "details": proc.stderr}), 500

    video_urls = [url.strip() for url in proc.stdout.split('\n') if url.strip()]
    video_urls = video_urls[:limit]
    
    results = []
    for i, video_url in enumerate(video_urls):
        print(f"\n🎬 [{i+1}/{len(video_urls)}] Обработка: {video_url}")
        try:
            # 1. Получаем информацию о видео (название и ID)
            info_cmd = [
                'yt-dlp', '--skip-download',
                '--print', '%(title)s',
                '--print', '%(id)s',
                '--user-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                '--extractor-args', 'youtube:player_client=android,web',
                video_url
            ]
            info_proc = subprocess.run(info_cmd, capture_output=True, text=True, timeout=30)
            if info_proc.returncode != 0:
                results.append({"url": video_url, "error": "Failed to get video info", "details": info_proc.stderr})
                continue

            lines = info_proc.stdout.strip().split('\n')
            video_title = lines[0] if len(lines) > 0 else f"video_{int(time.time())}"
            video_id = lines[1] if len(lines) > 1 else ""

            # Очищаем название для имени папки
            safe_title = re.sub(r'[\\/*?:"<>|]', "", video_title)
            safe_title = safe_title.strip()[:100]
            
            # Создаём папку для видео
            video_folder = os.path.join(downloads_dir, f"{safe_title}_{video_id}")
            os.makedirs(video_folder, exist_ok=True)
            print(f"   📁 Папка: {video_folder}")
            
            # СОЗДАЁМ ФАЙЛ title.txt С НАЗВАНИЕМ ВИДЕО
            try:
                with open(os.path.join(video_folder, 'title.txt'), 'w', encoding='utf-8') as f:
                    f.write(video_title)
                print(f"   📄 title.txt сохранён")
            except Exception as e:
                print(f"   ⚠️ Ошибка сохранения title.txt: {e}")
            
            # 2. Скачиваем субтитры (универсальная версия со всеми вариантами английского)
            subtitles_content = None
            subtitles_error = None
            
            # Все возможные варианты английского языка
            english_variants = ['en', 'en-US', 'en-GB', 'en-UK', 'en-AU', 'en-CA']
            
            # Пробуем сначала обычные субтитры, потом автоматические
            for autosubs in [False, True]:
                for lang in english_variants:
                    if subtitles_content:
                        break
                    tmp_dir = tempfile.mkdtemp()
                    try:
                        output_template = os.path.join(tmp_dir, 'subtitles.%(ext)s')
                        cmd_subs = [
                            'yt-dlp', '--skip-download',
                            '--write-subs' if not autosubs else '--write-auto-subs',
                            '--sub-lang', lang,
                            '--sub-format', 'srt',
                            '--output', output_template,
                            video_url
                        ]
                        subprocess.run(cmd_subs, capture_output=True, text=True, timeout=60)
                        
                        # Ищем файл субтитров
                        for ext in [f'{lang}.srt', 'srt', f'{lang}.vtt', 'vtt']:
                            fname = os.path.join(tmp_dir, f'subtitles.{ext}')
                            if os.path.exists(fname):
                                with open(fname, 'r', encoding='utf-8') as f:
                                    subtitles_content = f.read()
                                print(f"   📝 Субтитры найдены: {lang} (autosubs={autosubs})")
                                break
                    except Exception as e:
                        pass
                    finally:
                        shutil.rmtree(tmp_dir, ignore_errors=True)
            
            if not subtitles_content:
                # Проверяем, есть ли вообще субтитры на видео
                check_cmd = ['yt-dlp', '--list-subs', video_url]
                check_proc = subprocess.run(check_cmd, capture_output=True, text=True, timeout=30)
                if any(v in check_proc.stdout for v in english_variants):
                    subtitles_error = "Failed to extract subtitles (YouTube extraction error)"
                else:
                    subtitles_error = "No English subtitles available for this video"
                print(f"   ⚠️ {subtitles_error}")
            
            # Сохраняем субтитры, если есть
            if subtitles_content:
                with open(os.path.join(video_folder, 'subtitles.srt'), 'w', encoding='utf-8') as f:
                    f.write(subtitles_content)
                # TXT версия
                txt_lines = [l for l in subtitles_content.split('\n') if l.strip() and not l.strip().isdigit() and '-->' not in l]
                with open(os.path.join(video_folder, 'subtitles.txt'), 'w', encoding='utf-8') as f:
                    f.write('\n'.join(txt_lines))
            
            # 3. Скачиваем превью
            thumb_url = f"https://img.youtube.com/vi/{video_id}/mqdefault.jpg"
            try:
                r = requests.get(thumb_url, timeout=10)
                if r.status_code == 200:
                    with open(os.path.join(video_folder, 'thumbnail.jpg'), 'wb') as f:
                        f.write(r.content)
                    print(f"   🖼️ Превью сохранено")
                else:
                    print(f"   ⚠️ Превью не найдено")
            except Exception as e:
                print(f"   ⚠️ Превью: {e}")
            
            # 4. Скачиваем видео 720p
            tmp_video = f"/tmp/video_{video_id}.mp4"
            video_size = 0
            try:
                cmd_video = [
                    'yt-dlp',
                    '-f', 'bestvideo[height<=720]+bestaudio/best[height<=720]',
                    '--merge-output-format', 'mp4',
                    '--user-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
                    '--extractor-args', 'youtube:player_client=android,web',
                    '-o', tmp_video,
                    video_url
                ]
                proc_video = subprocess.run(cmd_video, capture_output=True, timeout=300)
                if proc_video.returncode == 0 and os.path.exists(tmp_video):
                    video_path = os.path.join(video_folder, 'video.mp4')
                    shutil.copy2(tmp_video, video_path)
                    video_size = round(os.path.getsize(video_path) / (1024 * 1024), 1)
                    print(f"   🎬 Видео сохранено ({video_size} MB)")
                else:
                    raise Exception("Video download failed")
            except Exception as e:
                print(f"   ⚠️ Видео: {e}")
                video_size = 0
            finally:
                if os.path.exists(tmp_video):
                    os.remove(tmp_video)
            
            results.append({
                "url": video_url,
                "title": video_title,
                "id": video_id,
                "folder": video_folder,
                "subtitles": subtitles_content is not None,
                "subtitles_error": subtitles_error,
                "thumbnail": os.path.exists(os.path.join(video_folder, 'thumbnail.jpg')),
                "video": os.path.exists(os.path.join(video_folder, 'video.mp4')),
                "size_mb": video_size
            })
            
            # Задержка между видео
            if i < len(video_urls) - 1:
                time.sleep(3)
            
        except Exception as e:
            results.append({"url": video_url, "error": str(e)})
            print(f"   ❌ Ошибка: {e}")
    
    return jsonify({
        "success": True,
        "total": len(video_urls),
        "processed": len([r for r in results if 'error' not in r]),
        "results": results
    })

@app.route("/status", methods=["GET", "OPTIONS"])
def status():
    return jsonify({"status": "running", "message": "YouTube Downloader Server"})

def run_server():
    app.run(host="0.0.0.0", port=9999, debug=False, threaded=True, use_reloader=False)

print("Остановка старых процессов...")
!pkill -f cloudflared 2>/dev/null || true
!fuser -k 9999/tcp 2>/dev/null || true

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)
print("✅ Сервер запущен на http://localhost:9999")

print("\n" + "=" * 80)
print("🌐 Настройка Cloudflare Tunnel")
print("=" * 80)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb 2>/dev/null || apt-get install -f -y
!cloudflared tunnel --url http://localhost:9999 > /tmp/cloudflared.log 2>&1 &
time.sleep(10)

public_url = None
with open('/tmp/cloudflared.log', 'r') as f:
    logs = f.read()
    matches = re.findall(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', logs)
    if matches:
        public_url = matches[-1]

if public_url:
    print(f"\n🔗 ПУБЛИЧНЫЙ URL: {public_url}")
    with open('/content/youtube_downloader_url.txt', 'w') as f:
        f.write(public_url)
    print("\n✅ Сервер готов. Используйте этот URL в вашей HTML‑странице.")
    print("\n📝 Примеры запросов:")
    print(f"  Субтитры: {public_url}/subtitles?url=YOUTUBE_URL&lang=en&format=srt&autosubs=true")
    print(f"  Превью:   {public_url}/thumbnail?url=YOUTUBE_URL")
    print(f"  Видео:    {public_url}/drive?url=YOUTUBE_URL")
    print(f"  Канал:    {public_url}/channel?url=https://www.youtube.com/@fallontonight/videos&limit=5")
else:
    print("\n❌ Не удалось получить URL. Проверьте логи выше.")
print("\n💡 Не закрывайте эту вкладку – сервер работает, пока выполняется ячейка.")

## 🔗 Использование

После запуска вы получите публичный URL вида `https://....trycloudflare.com`.

### Примеры запросов:

```
# Субтитры (SRT) – автоматически пробует все варианты английского (en, en-US, en-GB...)
GET {URL}/subtitles?url=YOUTUBE_URL&lang=en&format=srt&autosubs=true

# Превью (320x180)
GET {URL}/thumbnail?url=YOUTUBE_URL

# Видео на Google Drive (быстро, с названием видео)
GET {URL}/drive?url=YOUTUBE_URL

# Обработка канала – скачать последние 5 видео
GET {URL}/channel?url=https://www.youtube.com/@fallontonight/videos&limit=5
```

### Структура сохранения на Google Drive

```
yt_downloads/
├── Название видео 1_ID1/
│   ├── video.mp4
│   ├── subtitles.srt
│   ├── subtitles.txt
│   ├── thumbnail.jpg
│   └── title.txt          <-- файл с названием видео
├── Название видео 2_ID2/
│   ├── video.mp4
│   ├── subtitles.srt
│   ├── subtitles.txt
│   ├── thumbnail.jpg
│   └── title.txt
└── ...
```

## ⚠️ Примечания

- Сервер работает, пока открыта вкладка Colab. При закрытии или отключении Colab публичный URL станет недоступным.
- Для постоянной работы разверните код на своём сервере (VPS, Railway, Render) или используйте Google Drive для видео.
- YouTube может временно блокировать IP при частых запросах – подождите 10–15 минут или перезапустите Colab для получения нового IP.
- Субтитры скачиваются для видео, где они есть, с поддержкой всех вариантов английского языка (en, en-US, en-GB, en-UK, en-AU, en-CA).